# Others Analysis Test

이 노트북은 `classification_detail` 또는 `classification_full`에서 `pred_topic_type = 'others'`를 분석하여
- 그룹별 others 비중
- 반복 문구 기반 신규 topic 후보
를 확인하기 위한 테스트용입니다.


In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)


In [ ]:
%sh
cd /Workspace/Users/jungryo.lee@lge.com/prj_TV_voc && git pull

In [ ]:
import common.config_loader as config_loader
import taxonomy.others_analyzer as others_analyzer

importlib.reload(config_loader)
importlib.reload(others_analyzer)

from common.config_loader import load_config
from taxonomy.others_analyzer import analyze_others

config = load_config(f"{PROJECT_ROOT}/config/settings.yaml")


In [ ]:
MODEL_KEY = "gpt_55"
MODEL_VERSION_VALUE = config["llm"]["models"][MODEL_KEY]["model_version"]
# classification_detail: memo_id 기준 샘플/분류 검증용
# classification_full: 원본 row 기준 운영 분포용
CLASSIFICATION_INPUT_TABLE_KEY = "classification_full"
TARGET_CATE_1_DEPTH = "07. 스마트 사용성"
TARGET_CATE_2_DEPTH = "07-01. 채널/컨텐츠 APP"
TARGET_SC_MEASUREMENT = 1

analysis_result = analyze_others(
    spark=spark,
    config=config,
    input_table_key=CLASSIFICATION_INPUT_TABLE_KEY,
    cate_1_depth=TARGET_CATE_1_DEPTH,
    cate_2_depth=TARGET_CATE_2_DEPTH,
    sc_measurement=TARGET_SC_MEASUREMENT,
    model_version=MODEL_VERSION_VALUE,
    prompt_version=config["version"]["prompt_version"],
    taxonomy_version=config["version"]["taxonomy_version"],
)

print(analysis_result["analysis_config"])
print({
    "input_table_key": analysis_result["input_table_key"],
    "detail_cnt": analysis_result["detail_df"].count(),
    "others_cnt": analysis_result["others_df"].count(),
})

In [ ]:
display(analysis_result["others_group_summary_df"])

In [ ]:
display(
    analysis_result["new_topic_candidate_df"].select(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "memo_norm",
        "candidate_cnt",
        "candidate_ratio",
        "sample_memo",
        "candidate_reason",
    )
)


In [ ]:
display(
    analysis_result["others_df"].select(
        "memo",
        "pred_topic",
        "pred_topic_type",
        "classification_stage",
        "match_reason",
    )
    .orderBy(F.rand(42))
    .limit(30)
)